# **Distributed Big Data Sentiment Mining System**
## Part 1 — Data Ingestion & Initial Inspection

**Dataset:** Amazon Reviews 2023 — Automotive Category (McAuley Lab / HuggingFace)  
**Task:** Binary Sentiment Classification (Positive / Negative)  
**Scale:** 19,955,450 reviews · 8.3 GB uncompressed  
**Source:** https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023

# **1. Install Dependencies for PySpark**
This section installs the required system and Python dependencies for Apache PySpark, including Java (OpenJDK 17), PySpark, and findspark, to enable proper Spark initialization in the notebook environment.

In [2]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install -q -U "pyspark[connect]~=4.0.0" findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.3/434.3 MB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


# **2. SparkSession Initialisation**

This section initializes the SparkSession, the main entry point for Spark, and sets key configurations such as JAVA_HOME, driver memory, and shuffle partitions for efficient local execution.

In [3]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Automotive_Sentiment_Data_Ingestion")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

# **3. Download Dataset**

This section downloads the Amazon Automotive JSONL dataset from Hugging Face and saves it locally for efficient loading and processing with Spark.

In [ ]:
!wget -q --show-progress \
    "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Automotive.jsonl" \
    -O /content/Automotive.jsonl

print("Download complete.")

/content/Automotive 100%[===================>]   8.13G  65.3MB/s    in 2m 10s  
Download complete.


# **4. File Size Verification**

This section verifies the dataset download by checking file size confirming it is large enough to require distributed PySpark processing rather than single machine tools like Pandas.

In [ ]:
import os

path = "/content/Automotive.jsonl"
print(f"File exists : {os.path.exists(path)}")
print(f"File size   : {os.path.getsize(path) / (1024 * 1024):.1f} MB")

File exists : True
File size   : 8323.4 MB


# **5. Dataset Loading**


In [ ]:
df = spark.read.json("/content/Automotive.jsonl")

# **6. Row and Column Count**

This section calculates the total number of rows and columns in the dataset and provides an overview of its scale and suitability.

In [ ]:
print("Number of rows    :", df.count())
print("Number of columns :", len(df.columns))


Number of rows    : 19955450
Number of columns : 10


# **7. Schema Inspection**

In [ ]:
df.printSchema()

root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



# **8. View First 20 rows**

In [ ]:
df.show(20, truncate=False)

+----------+------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# **9. Spark Session Termination**

This section terminates the active SparkSession to release allocated memory and cluster resources. Properly closing the session prevents resource leaks and ensures a clean environment for future runs.





In [ ]:
spark.stop()
print("Spark session stopped successfully.")

Spark session stopped successfully.


# **Data Ethics and Licensing**

# **Data Ethics**

This project uses the publicly available Amazon Reviews 2023 dataset, which provides large scale customer review text for research and non-commercial analysis of Natural Language Processing and Big Data techniques.

The dataset does not include personal or identifiable information, and no effort is made to link reviews back to real individuals. Any user IDs that appear in the data are anonymous, system generated placeholders with no connection to actual users.

A few ethical considerations apply when working with this type of material. Not all reviews reflect genuine experiences, and certain opinions or sentiments may appear more frequently than others. The data may also show patterns influenced by broader social or behavioural trends. Since the dataset contains a large volume of user‑generated text, it is handled with care and respect throughout this project.

# **Dataset Licensing**

The Amazon Reviews 2023 dataset is released for non‑commercial academic use and can be accessed through both the McAuley Lab repository and the Hugging Face platform.

Source: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023
